In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import geopandas as gpd
import os
import xarray as xr
from tqdm.auto import tqdm

## Define paths in directory

In [ ]:
# Path to Hubbard data for convenience
data_path = os.getcwd()
# Centerline
cl_fn = os.path.join(data_path, '../data/velocity', 'center_50m.gpkg')

## Load data

In [ ]:
# Centerline
cl = gpd.read_file(cl_fn)
# calculate distance vector
def create_distance_vector(line):
    x, y = line.coords.xy[0], line.coords.xy[1]
    line_dist = np.zeros(len(line.coords.xy[0]))
    for i in range(1, len(line.coords.xy[0])):
        line_dist[i] = np.sqrt((x[i]-x[i-1])**2 + (y[i]-y[i-1])**2) + line_dist[i-1]
    return line_dist
cl_dist = create_distance_vector(cl.geometry[0])
# Flip to start near terminus
cl_dist = np.flip(cl_dist)
x_samp, y_samp = cl.geometry[0].coords.xy

In [ ]:
# -----Calculate summer and winter speed speaks
# Load velocity
ds_fn = os.path.join(data_path, '../data/velocity', 'Hubbard_S1.nc')
ds = xr.load_dataset(ds_fn)
ds["v"] = np.sqrt(ds.vx**2 + ds.vy**2).fillna(0)
ds["v_smooth"] = ds["v"].rolling(time=5, center=True).mean()
ds["v_grad"] = ds.v_smooth.differentiate("time")
ds["month"] = ds.time.dt.month
ds["year"] = ds.time.dt.year
ds["doy"] = ds.time.dt.dayofyear

## Calculate speed peak strengths along centerline

Copied from Amy's double_peak_strength notebook via surface_elevations_centerline

In [ ]:
# Calculate strength of speed peaks
strengths_cl = pd.DataFrame()
strengths_norm_cl = pd.DataFrame()
years = [2016, 2017, 2018, 2019, 2020, 2021]
fig, ax = plt.subplots()
i = 0
for year in tqdm(years):

    # Winter peak
    winter_mask = np.logical_or(
        np.logical_and(ds.month >= 11, ds.year == year),
        np.logical_and(ds.month <= 2, ds.year == year+1)
    )
    winter_velocities = ds.v[winter_mask, :, :]
    winter_peak = winter_velocities.max(dim='time')

    # Middle peak
    middle_mask = np.logical_and(
        np.logical_and(ds.month >= 2.05, ds.month <= 4),
        ds.year == year+1,
    )
    middle_velocities = ds.v[middle_mask, :, :]
    middle_min = middle_velocities.min(dim='time')

    # Summer peak
    summer_mask = np.logical_and(
        np.logical_and(ds.month >= 4, ds.month <= 6),
        ds.year == year+1,
    )
    summer_velocities = ds.v[summer_mask, :, :]
    summer_peak = summer_velocities.max(dim='time')

    # Strength of double peaks 
    avg_mag = (winter_peak + summer_peak)/2
    mask = (winter_peak < middle_min) | (summer_peak < 500)
    strength = (winter_peak - middle_min) 
    strength = xr.where(mask, np.nan, strength)

    # Sample along centerline
    strength_cl_np = [float(strength.sel(x=x, y=y, method='nearest').data) for (x, y) in list(zip(x_samp, y_samp))]
    strength_cl = pd.DataFrame({year: strength_cl_np})

    # Smooth over a 500 m window
    strength_cl_smooth = strength_cl.rolling(10).mean()
    strengths_cl = pd.concat([strengths_cl, strength_cl_smooth], axis=1)
    ax.plot(strengths_cl[year], '-', color=plt.cm.viridis(i/len(years)))
    i+=1
    
plt.show() 

In [ ]:
filled_da = winter_velocities.fillna(-9999) 
winter_peak_time = filled_da.argmax(dim='time', skipna=None)
# Sample along centerline
timing_cl_np = [float(winter_peak_time.sel(x=x, y=y, method='nearest').data) for (x, y) in list(zip(x_samp, y_samp))]
timing_cl = pd.DataFrame({year: timing_cl_np})
plt.plot(timing_cl)

In [ ]:
for i, year in enumerate([2016, 2017, 2018, 2019, 2020, 2021]):
    mask = np.logical_or(
        np.logical_and(ds.month >= 11, ds.year == year),
        np.logical_and(ds.month <= 2, ds.year == year+1)
    )
    peakdoy = ds.doy[mask][ds.v[mask, :, :].argmax(dim="time", skipna=True)].to_numpy().astype(np.float32)

    # If max value is at first day then no peak
    peakdoy[peakdoy == 306] = np.nan
    peakdoy[peakdoy == 305] = np.nan

    peakdoy[peakdoy > 180] = peakdoy[peakdoy > 180] - 365
